In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/cleaned_data.csv')
print("Shape:", df.shape)

Shape: (101766, 43)


In [2]:
# Feature 1 — Service Utilization Score
df['service_utilization'] = (
    df['number_outpatient'] + 
    df['number_emergency'] + 
    df['number_inpatient']
)

# Feature 2 — Medication Procedure Complexity
df['medication_procedure_complexity'] = (
    df['num_medications'] * 
    df['num_procedures']
)

print("New features added. Shape:", df.shape)
print("\nService Utilization Sample:")
print(df['service_utilization'].describe())
print("\nMedication Complexity Sample:")
print(df['medication_procedure_complexity'].describe())

New features added. Shape: (101766, 45)

Service Utilization Sample:
count    101766.000000
mean          1.202759
std           2.291781
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max          80.000000
Name: service_utilization, dtype: float64

Medication Complexity Sample:
count    101766.000000
mean         26.813199
std          44.869480
min           0.000000
25%           0.000000
50%          10.000000
75%          36.000000
max         450.000000
Name: medication_procedure_complexity, dtype: float64


In [3]:
print("Correlation with target BEFORE engineering:")
print(df[['number_outpatient', 'number_emergency', 
          'number_inpatient', 'num_medications']].corrwith(df['readmitted']))

print("\nCorrelation with target AFTER engineering:")
print(df[['service_utilization', 
          'medication_procedure_complexity']].corrwith(df['readmitted']))

Correlation with target BEFORE engineering:
number_outpatient    0.018893
number_emergency     0.060747
number_inpatient     0.165147
num_medications      0.038432
dtype: float64

Correlation with target AFTER engineering:
service_utilization                0.126114
medication_procedure_complexity    0.000699
dtype: float64


In [4]:
# Drop the failed engineered feature

df = df.drop(columns=['medication_procedure_complexity'])
print("Shape after dropping failed feature:", df.shape)

Shape after dropping failed feature: (101766, 44)


In [5]:
# Separate categorical and numerical columns
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target from numerical list
numerical_cols.remove('readmitted')

print(f"Categorical columns ({len(categorical_cols)}):")
print(categorical_cols)
print(f"\nNumerical columns ({len(numerical_cols)}):")
print(numerical_cols)

Categorical columns (31):
['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']

Numerical columns (12):
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'service_utilization']


In [6]:
# unique values in each categorical column
print("Unique values per categorical column:")
for col in categorical_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")

Unique values per categorical column:

race: 6 unique values

gender: 3 unique values

age: 10 unique values

diag_1: 717 unique values

diag_2: 749 unique values

diag_3: 790 unique values

max_glu_serum: 3 unique values

A1Cresult: 3 unique values

metformin: 4 unique values

repaglinide: 4 unique values

nateglinide: 4 unique values

chlorpropamide: 4 unique values

glimepiride: 4 unique values

acetohexamide: 2 unique values

glipizide: 4 unique values

glyburide: 4 unique values

tolbutamide: 2 unique values

pioglitazone: 4 unique values

rosiglitazone: 4 unique values

acarbose: 4 unique values

miglitol: 4 unique values

troglitazone: 2 unique values

tolazamide: 3 unique values

insulin: 4 unique values

glyburide-metformin: 4 unique values

glipizide-metformin: 2 unique values

glimepiride-pioglitazone: 2 unique values

metformin-rosiglitazone: 2 unique values

metformin-pioglitazone: 2 unique values

change: 2 unique values

diabetesMed: 2 unique values


ICD Code Range   →    Disease Category
***
001–139          →    Infectious Disease\
140–239          →    Cancer \
240–279          →    Diabetes/Metabolic \
280–289          →    Blood Disease \
290–319          →    Mental Health \
320–389          →    Nervous System \
390–459          →    Circulatory \
460–519          →    Respiratory \
520–579          →    Digestive \
580–629          →    Genitourinary \
630–679          →    Pregnancy \
680–709          →    Skin Disease \
710–739          →    Musculoskeletal \
740–759          →    Congenital \
760–799          →    Perinatal/Symptoms \
800–999          →    Injury/Poisoning \
E/V codes        →    External/Supplementary

In [7]:
def group_icd_codes(code):
    """
    Groups ICD-9 diagnosis codes into broader disease categories.
    Returns 'Other' for codes that don't match known patterns.
    """
    # Handle missing or non-standard codes
    if pd.isna(code):
        return 'Other'
    
    code = str(code)
    
    # E and V codes — External causes and supplementary
    if code.startswith('E') or code.startswith('V'):
        return 'External_Supplementary'
    
    try:
        code_num = float(code)
        
        if 1 <= code_num <= 139:
            return 'Infectious'
        elif 140 <= code_num <= 239:
            return 'Cancer'
        elif 240 <= code_num <= 279:
            return 'Diabetes_Metabolic'
        elif 280 <= code_num <= 289:
            return 'Blood_Disease'
        elif 290 <= code_num <= 319:
            return 'Mental_Health'
        elif 320 <= code_num <= 389:
            return 'Nervous_System'
        elif 390 <= code_num <= 459:
            return 'Circulatory'
        elif 460 <= code_num <= 519:
            return 'Respiratory'
        elif 520 <= code_num <= 579:
            return 'Digestive'
        elif 580 <= code_num <= 629:
            return 'Genitourinary'
        elif 630 <= code_num <= 679:
            return 'Pregnancy'
        elif 680 <= code_num <= 709:
            return 'Skin_Disease'
        elif 710 <= code_num <= 739:
            return 'Musculoskeletal'
        elif 740 <= code_num <= 759:
            return 'Congenital'
        elif 760 <= code_num <= 799:
            return 'Symptoms'
        elif 800 <= code_num <= 999:
            return 'Injury_Poisoning'
        else:
            return 'Other'
            
    except ValueError:
        return 'Other'

In [8]:
# Apply ICD grouping
df['diag_1'] = df['diag_1'].apply(group_icd_codes)
df['diag_2'] = df['diag_2'].apply(group_icd_codes)
df['diag_3'] = df['diag_3'].apply(group_icd_codes)

# Verify reduction in unique values
print("Unique values after grouping:")
print(f"diag_1: {df['diag_1'].nunique()} categories")
print(f"diag_2: {df['diag_2'].nunique()} categories")
print(f"diag_3: {df['diag_3'].nunique()} categories")

print("\ndiag_1 value counts:")
print(df['diag_1'].value_counts())

Unique values after grouping:
diag_1: 18 categories
diag_2: 18 categories
diag_3: 18 categories

diag_1 value counts:
diag_1
Circulatory               30336
Diabetes_Metabolic        11459
Respiratory               10407
Digestive                  9208
Symptoms                   7636
Injury_Poisoning           6974
Genitourinary              5078
Musculoskeletal            4957
Cancer                     3433
Infectious                 2768
Skin_Disease               2530
Mental_Health              2262
External_Supplementary     1645
Nervous_System             1211
Blood_Disease              1103
Pregnancy                   687
Congenital                   51
Other                        21
Name: count, dtype: int64


In [9]:
# Label encoding for age as it contains order

from sklearn.preprocessing import LabelEncoder

ordinal_cols = ['age']

le = LabelEncoder()
df['age'] = le.fit_transform(df['age'])

print("Age encoding mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls} : {i}")

Age encoding mapping:
  [0-10) : 0
  [10-20) : 1
  [20-30) : 2
  [30-40) : 3
  [40-50) : 4
  [50-60) : 5
  [60-70) : 6
  [70-80) : 7
  [80-90) : 8
  [90-100) : 9


In [10]:
# These have no natural order
nominal_cols = [
    'race', 'gender', 'insulin', 
    'diag_1', 'diag_2', 'diag_3',
    'max_glu_serum', 'A1Cresult',
    'change', 'diabetesMed'
]

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print("Shape after encoding:", df.shape)
print("Sample of new columns:")
print(df.columns.tolist()[:20])

Shape after encoding: (101766, 101)
Sample of new columns:
['age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide']


In [11]:
# Check if any object columns remain
remaining_object_cols = df.select_dtypes(include='object').columns.tolist()
print("Remaining object columns:", remaining_object_cols)

# Drop any that remain
if remaining_object_cols:
    df = df.drop(columns=remaining_object_cols)
    print("Dropped:", remaining_object_cols)

print("\nFinal shape:", df.shape)
print("All dtypes object?", df.select_dtypes(include='object').shape[1] == 0)

Remaining object columns: ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
Dropped: ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

Final shape: (101766, 81)
All dtypes object? True


In [12]:
# Find and fix remaining object columns
remaining_object_cols = df.select_dtypes(include='object').columns.tolist()
print("Remaining object columns:", remaining_object_cols)

# Check unique values for each
for col in remaining_object_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts())

Remaining object columns: []


In [13]:
# Full preprocessing sanity check

print("Dataset Shape:", df.shape)
print("\nAny object columns remaining:", 
      df.select_dtypes(include='object').shape[1])
print("\nAny missing values:", df.isnull().sum().sum())
print("\nTarget distribution:")
print(df['readmitted'].value_counts())
print("\nSample of columns:")
print(df.columns.tolist()[:15])

Dataset Shape: (101766, 81)

Any object columns remaining: 0

Any missing values: 0

Target distribution:
readmitted
0    90409
1    11357
Name: count, dtype: int64

Sample of columns:
['age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'readmitted', 'service_utilization', 'race_AfricanAmerican']


In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=['readmitted'])
y = df['readmitted']

# Split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # Important for imbalanced data
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nTrain target distribution:")
print(y_train.value_counts())
print("\nTest target distribution:")
print(y_test.value_counts())

X_train shape: (81412, 80)
X_test shape: (20354, 80)

Train target distribution:
readmitted
0    72326
1     9086
Name: count, dtype: int64

Test target distribution:
readmitted
0    18083
1     2271
Name: count, dtype: int64


In [15]:
from sklearn.preprocessing import StandardScaler

numerical_cols = [
    'age', 'time_in_hospital', 'num_lab_procedures',
    'num_procedures', 'num_medications', 'number_outpatient',
    'number_emergency', 'number_inpatient', 'number_diagnoses',
    'service_utilization'
]

# Fit ONLY on training data
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])

# Apply to test data — no fitting here
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("Scaling complete!")
print("X_train sample after scaling:")
print(X_train[numerical_cols].describe().round(2))

Scaling complete!
X_train sample after scaling:
            age  time_in_hospital  num_lab_procedures  num_procedures  \
count  81412.00          81412.00            81412.00        81412.00   
mean      -0.00              0.00               -0.00            0.00   
std        1.00              1.00                1.00            1.00   
min       -3.82             -1.14               -2.14           -0.79   
25%       -0.69             -0.80               -0.61           -0.79   
50%       -0.06             -0.13                0.05           -0.20   
75%        0.57              0.54                0.71            0.39   
max        1.82              3.22                4.51            2.74   

       num_medications  number_outpatient  number_emergency  number_inpatient  \
count         81412.00           81412.00          81412.00          81412.00   
mean              0.00               0.00              0.00             -0.00   
std               1.00               1.00          

In [16]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(y_train.value_counts())
print("\nAfter SMOTE:")
print(pd.Series(y_train_resampled).value_counts())
print("\nX_train shape after SMOTE:", X_train_resampled.shape)

Before SMOTE:
readmitted
0    72326
1     9086
Name: count, dtype: int64

After SMOTE:
readmitted
0    72326
1    72326
Name: count, dtype: int64

X_train shape after SMOTE: (144652, 80)


In [17]:
import joblib

# Save processed datasets
X_train_resampled.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
pd.Series(y_train_resampled).to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save scaler for deployment
joblib.dump(scaler, '../models/scaler.pkl')

print("All processed data saved!")
print("Scaler saved to models/scaler.pkl")

All processed data saved!
Scaler saved to models/scaler.pkl
